## Codificar programas faltantes de las bases  de censos universitarios y TUNI

In [14]:
# Instalar librerías
# !pip install -q pandas numpy scikit-learn sentence-transformers torch

In [15]:
# Paquetes
from pathlib import Path
import pandas as pd
import numpy as np
import re
import unicodedata

from sklearn.neighbors import NearestNeighbors
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score

In [16]:
# Cargar entrenamiento
carpeta = Path(
    r"D:\Daniel_Pajita\1_Proyectos\Research\Educacion\educacion-universitaria-peru\2_Datos\Correspondencias_entre_clasificaciones"
)

df_train = pd.read_csv(carpeta / "BD_tuni_programas_entrenamiento.txt",sep="|",dtype=str,encoding="utf-8-sig",)

# Combinar los programas pendientes de clasificación
archivos_prediccion = ["BD_tuni_programas_predicción.txt", "programas_censo1996.txt", "programas_censo2010.txt",]

df_pred = pd.concat(
    [
        pd.read_csv(carpeta / archivo,sep="|",dtype=str,encoding="utf-8-sig",)
        for archivo in archivos_prediccion
    ],
    ignore_index=True,
)

df_pred = (df_pred.drop_duplicates(subset=["nivel_academico", "nombre_programa"]).reset_index(drop=True))

print(f"Entrenamiento: {len(df_train):,} programas")
print(f"Predicción: {len(df_pred):,} programas")

Entrenamiento: 5,122 programas
Predicción: 13,000 programas


In [17]:
# Limpieza 
def clean_text(s):
    if pd.isna(s):
        return ""
    s = str(s).strip().lower()
    s = re.sub(r"\s+", " ", s)
    return s

for df in [df_train, df_pred]:
    df["nivel_academico_clean"] = df["nivel_academico"].apply(clean_text)
    df["nombre_programa_clean"] = df["nombre_programa"].apply(clean_text)

# Texto final que usará el modelo (concatena nivel + nombre)
df_train["texto_modelo"] = df_train["nivel_academico_clean"] + " | " + df_train["nombre_programa_clean"]
df_pred["texto_modelo"]  = df_pred["nivel_academico_clean"]  + " | " + df_pred["nombre_programa_clean"]

In [18]:
# Modelo multilingüe MiniLM para embeddings
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [19]:
# Crear embeddings del entrenamiento (solo una vez)
X_train = df_train["texto_modelo"].tolist()
emb_train = model.encode(X_train, show_progress_bar=True, normalize_embeddings=True)

Batches:   0%|          | 0/161 [00:00<?, ?it/s]

In [20]:
# Clasificar por similitud (KNN por vecinos más cercanos)
nn = NearestNeighbors(n_neighbors=1, metric="cosine")
nn.fit(emb_train)

X_pred = df_pred["texto_modelo"].tolist()
emb_pred = model.encode(X_pred, show_progress_bar=True, normalize_embeddings=True)
dist, idx = nn.kneighbors(emb_pred)  
df_pred["match_score"] = 1 - dist.flatten() 
df_pred["match_index"] = idx.flatten()

Batches:   0%|          | 0/407 [00:00<?, ?it/s]

In [21]:
# Completar las columnas desde el match
cols_to_copy = ["codigo_grupo_1", "nombre_grupo_1", "codigo_grupo_3", "nombre_grupo_3"]
for c in cols_to_copy:
    df_pred[c] = df_train.iloc[df_pred["match_index"]][c].values

In [22]:
# Marcar para revisión manual
umbral = 0.95
df_pred["revision_manual"] = (df_pred["match_score"] < umbral).astype(int)
df_pred[df_pred["revision_manual"]==1][["nivel_academico","nombre_programa","match_score"] + cols_to_copy].head(100)

,nivel_academico,nombre_programa,match_score,codigo_grupo_1,nombre_grupo_1,codigo_grupo_3,nombre_grupo_3
1,Pregrado,"Administración Hotelera, Turismo, Ecoturismo y...",0.949243,0,Servicios,12,"Hotelería, Restaurantes y Gastronomía"
2,Pregrado,Administración de Redes y Seguridad Informática,0.815914,6,Tecnología de la Información y la Comunicación,612,Sistemas y Cómputo
5,Pregrado,Administración y Negocios Internacionales Moda...,0.743357,4,Ciencias Administrativas y Derecho,416,Negocios Internacionales
23,Pregrado,Carrera Profesional de Ciencias Administrativa...,0.834999,4,Ciencias Administrativas y Derecho,413,Gestión y Administración
24,Pregrado,Carrera Profesional de Ciencias de la Educació...,0.892075,1,Educación,112,Educación Primaria
...,...,...,...,...,...,...,...
231,Maestría,Máster en Dirección de Empresas A Tiempo Parci...,0.780302,4,Ciencias Administrativas y Derecho,419,Otros Programas en Administración y Comercio
232,Pregrado,Negocios Globales,0.933447,4,Ciencias Administrativas y Derecho,416,Negocios Internacionales
233,Pregrado,Programa De Complementación Pedagógica (PCP),0.848262,1,Educación,199,Otros Programas en Educación
234,Pregrado,Programa De Complementación Pedagógica Y Unive...,0.926979,1,Educación,199,Otros Programas en Educación


In [23]:
# Estandarizar nombres de programas
def estandarizar_programa(texto):
    if pd.isna(texto):
        return pd.NA

    texto = unicodedata.normalize("NFD", str(texto).upper())
    texto = "".join(
        c if c.isalnum() else " "
        for c in texto
        if unicodedata.category(c) != "Mn"
    )
    return " ".join(texto.split())


# Conservar el nombre original y estandarizar el campo de salida
df_pred["nombre_programa_original"] = df_pred["nombre_programa"]
df_pred["nombre_programa"] = (df_pred["nombre_programa"].map(estandarizar_programa))

In [24]:
# Número de repeticiones adicionales por programa y nivel
df_pred["dup"] = (df_pred.groupby(["nombre_programa", "nivel_academico"], dropna=False,)["nombre_programa"].transform("size").sub(1))

# Conservar la clasificación con mayor similitud
df_pred["match_score"] = pd.to_numeric(df_pred["match_score"], errors="coerce")
df_pred = (df_pred.sort_values(
        ["nivel_academico", "nombre_programa", "match_score"],ascending=[True, True, False],na_position="last",kind="stable",
    )
    .drop_duplicates(subset=["nivel_academico", "nombre_programa"],keep="first",)
    .reset_index(drop=True)
)
df_pred = df_pred.drop(columns="dup")

In [25]:
# Guardar
salida = carpeta / "BD_programas_prediccion_clasificada.txt"
df_pred.to_csv(salida, sep="|", index=False, encoding="utf-8")